In [9]:
import os
import streamlit as st
import google.generativeai as genai
from langchain.vectorstores import FAISS
from langchain.document_loaders import DirectoryLoader, PyPDFLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.prompts.chat import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain.schema import (
    AIMessage,
    HumanMessage,
    SystemMessage
)

In [10]:
class GeminiClientWrapper:
    def __init__(self, api_key):
        self.client = genai.Client(api_key=api_key)
    
    def generate(self, prompt: str):
        response = self.client.models.generate_content(
            model="gemini-2.0-flash", contents=prompt
        )
        return response.text

def setup_page():
    st.set_page_config(layout="wide")
    hide = """
    <style>
    MainMenu {visibility:hidden;}
    header {visibility:hidden;}
    footer {visibility:hidden;}
    </style>
    """
    st.markdown(hide, unsafe_allow_html=True)

def setup_session(session):
    if 'transcript' not in session:
        session.transcript = []
    if 'input_disabled' not in session:
        session.input_disabled = True
    if 'analyze_disabled' not in session:
        session.analyze_disabled = False
    if 'institute' not in session:
        session.institute = ""

def setup_llm():
    api_key = st.secrets["GEMINI_API_KEY"]
    llm = GeminiClientWrapper(api_key=api_key)
    return llm

def load_doc(path):
    k = 300000
    if path.endswith(".pdf"):
        doc = PyPDFLoader(file_path=path)
    else:
        doc = DirectoryLoader(path=path, glob="**/*.pdf")
    document = doc.load()
    context = "\n\n".join([document[i].page_content for i in range(len(document))])
    return context[:k]

def compare_answer(llm, session, question, docs):
    summaries = {}
    for doc_name, doc_txt in docs.items():
        prompt = f"Extract relevant information from the following text to answer the question: '{question}'\n\nContext:\n{doc_txt}"
        summaries[doc_name] = llm.generate(prompt)

    compare_context = "\n\n".join([f"Relevant points from {doc_name}:\n\n{summary}" for doc_name, summary in summaries.items()])
    details = "\n\n" + question + "\n\n" + compare_context

    final_prompt = f"""
You are a Reg Reporting Assistant helping answer a question based on multiple documents for the institute {session.institute}.
Below is a summary of relevant points extracted from each document. Answer the question clearly and concisely using the content below.
Question: {question}
{compare_context}
"""
    response = llm.generate(final_prompt)
    return details, response
